# Custom prediction with a trained model

This notebooks shows how to use a fine-tuned model for prediction tasks.

## Imports

In [ ]:
import os
# Set the GPU to the one with available memory (nvidia-smi)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from esnlir.dataset_utils.dataset import BERTDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd

In [ ]:
import os
os.getcwd()


## Constants

In [ ]:
# File path of the dataset you want to predict
DATASET = "data/test_full.jsonl"

# Folder path of the model you fine-tuned
MODEL = "Flaglab/ESNLIR-AL-BERTIN-LER"

In [ ]:
# Maximum length of tokens per sentence
MAX_LEN = 256

# Original model from hugging-faces
MODEL_TYPE = "bertin-project/bertin-roberta-base-spanish"

# Batch size
BATCH_SIZE = 32

# CUDA device
DEVICE = "cuda:0"

# CPU device
CPU_DEVICE = "cpu"

## Execution

### Load the dataset

In [ ]:
dataset = BERTDataset(
    dataframe_file=DATASET,
    max_len=MAX_LEN,
    model_type=MODEL_TYPE,
    only_premise=False,
    max_samples=None
)

In [ ]:
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

## Load the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(DEVICE)

## Predict the dataset with the model

In [ ]:
#
model.zero_grad()

# Real labels accumulator
y_true = []
y_pred = []
# Run over all dataset batches
for batch in tqdm(dataloader):
    
    # The model tokenized input
    inputs = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    
    # Get the labels to evaluate
    batch_y_true = batch["labels"].to(CPU_DEVICE).detach().numpy().tolist()
    y_true.extend(batch_y_true)
    
    # Predict using the model
    batch_y_pred = model(inputs, attention_mask=attention_mask).logits.to(CPU_DEVICE).detach().numpy().tolist()
    y_pred.extend(batch_y_pred)
    
    torch.cuda.empty_cache()

In [ ]:
y_true[:2]

In [ ]:
y_pred[:2]

## Generating metrics over predictions

As shown above, the predictions and real values are set as logits, meaning a vector of probability for each class. For each example, the positions are sorted by class name in order.

In [ ]:
dataset.classes

We can use the dataset labels to revert the logits to the original labels and use any standard metric library to evaluate the results

In [ ]:
class_map = {index: label for index, label in enumerate(dataset.classes)}
class_map

In [ ]:
# For each example find the biggest probability position and map it to the class name
y_true_cat = np.vectorize(class_map.get)(np.argmax(y_true, axis=1))
y_true_cat[:2]

In [ ]:
# For each example find the biggest probability position and map it to the class name
y_pred_cat = np.vectorize(class_map.get)(np.argmax(y_pred, axis=1))
y_pred_cat[:2]

## We can use sklearn over these new real and prediction label arrays

### Confusion matrix

In [ ]:
conf_mat = confusion_matrix(y_true_cat, y_pred_cat)

plt.figure()
plt.title("Demo confusion matrix")
ax = plt.gca()
sns.heatmap(conf_mat, annot=True, cmap="mako", ax=ax, xticklabels=dataset.classes, yticklabels=dataset.classes)
plt.show()

## Classification report

In [ ]:
class_report = classification_report(y_true_cat, y_pred_cat, output_dict=True)
df_class_metrics = pd.DataFrame(class_report)
df_class_metrics

In [ ]:
# Load raw test JSONL into a DataFrame for slicing
import pandas as pd
from pathlib import Path

# Use the configured DATASET path; supports JSONL
data_path = Path(DATASET)
if not data_path.exists():
    # fallback to local relative path
    data_path = Path("data/test.jsonl")


# Read JSON Lines (one JSON object per line)
try:
    df_test = pd.read_json(data_path, lines=True)
except ValueError:
    # If it's a regular JSON array file, fallback
    df_test = pd.read_json(data_path)
df_test.head(3)

In [ ]:
# Attach categorical true/pred labels to the DataFrame
df_test = df_test.assign(y_true=y_true_cat, y_pred=y_pred_cat)
df_test[['connector_type','genre','domain','y_true','y_pred']].head(5)

In [ ]:
# Helper: compute classification report per group (includes accuracy)
from sklearn.metrics import accuracy_score

def grouped_classification_report(df, group_col):
    results = []
    for grp_val, df_grp in df.groupby(group_col):
        if len(df_grp) == 0:
            continue
        # Per-group accuracy
        acc = accuracy_score(df_grp['y_true'], df_grp['y_pred'])
        # Detailed per-class metrics
        report = classification_report(
            df_grp['y_true'], df_grp['y_pred'], output_dict=True, zero_division=0
        )
        # Flatten report dict to rows
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                row = {'group': group_col, 'group_value': grp_val, 'label': label}
                row.update(metrics)
                # attach accuracy for convenience
                row['accuracy'] = acc
                results.append(row)
        # Add an aggregate accuracy row
        results.append({
            'group': group_col,
            'group_value': grp_val,
            'label': 'accuracy',
            'precision': None,
            'recall': None,
            'f1-score': None,
            'support': int(len(df_grp)),
            'accuracy': acc
        })
    return pd.DataFrame(results)

cols = ['connector_type','genre','domain']
group_reports = {c: grouped_classification_report(df_test, c) for c in cols}
{k: v.head(5) for k, v in group_reports.items()}

In [ ]:
# Save CSVs under models/xlmroberta/test/<group>/ with accuracy included
from pathlib import Path

base_dir = Path("models/bertin_active_Rem/test")
for group_name, df_metrics in group_reports.items():
    out_dir = base_dir / group_name
    out_dir.mkdir(parents=True, exist_ok=True)
    # General stats includes per-class rows and an extra 'accuracy' row
    df_metrics.to_csv(out_dir / "general_stats.csv", index=False)
    # Class-specific accuracy table (excludes the aggregate 'accuracy' row)
    class_acc = df_metrics[df_metrics['label'].isin(df_test['y_true'].unique())][['group','group_value','label','precision','recall','f1-score','support','accuracy']]
    class_acc.to_csv(out_dir / "class_accuracy.csv", index=False)
{g: str((base_dir / g).resolve()) for g in group_reports.keys()}

In [ ]:
# Compute per-genre overall accuracy and F1 (macro, weighted)
from sklearn.metrics import accuracy_score, f1_score

def genre_overall_metrics(df):
    rows = []
    for genre, df_g in df.groupby('genre'):
        if len(df_g) == 0:
            continue
        acc = accuracy_score(df_g['y_true'], df_g['y_pred'])
        f1_macro = f1_score(df_g['y_true'], df_g['y_pred'], average='macro')
        f1_weighted = f1_score(df_g['y_true'], df_g['y_pred'], average='weighted')
        rows.append({'genre': genre, 'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted, 'support': int(len(df_g))})
    return pd.DataFrame(rows).sort_values('genre')

genre_summary = genre_overall_metrics(df_test)
genre_summary

In [ ]:
# Save per-genre summary to CSV
from pathlib import Path
out_path = Path("models/bertin_active_Rem/test/genre/general_stats.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
genre_summary.to_csv(out_path, index=False)
out_path.as_posix()